# NLLB Translation Fine-Tuning & Evaluation Pipeline

This notebook provides an end-to-end pipeline to fine-tune a No Language Left Behind (NLLB) model (specifically `facebook/nllb-200-distilled-1.3B`), evaluate its performance on a test set, export it to the CTranslate2 format, and use it to translate formatted structures (including JSON, HTML, and text) using segment-aware planning.

---

## 🛠️ How to Adapt this Notebook for Your Language Pair

This notebook has been refactored to be **fully generic**. To adapt this pipeline to any other language pair, you only need to modify the configurations in the **second cell**:

### 1. Set Language Identifiers
Change the source and target NLLB language codes (`SRC_LANG` and `TGT_LANG`):

`SRC_LANG, TGT_LANG = "eng_Latn", "lij_Latn"`  *(Replace with your NLLB-200 language codes)*

### 2. Prepare Your Drive & Dataset
- Ensure you have mounted Google Drive.
- Store your datasets in Google Drive under a directory named exactly using your target language suffixes (e.g. `nllb_eng_lij`).
- The pipeline expects the following files inside your Drive directory:
  * `train.src_suffix` (e.g., `train.eng`): Your source language training file.
  * `train.tgt_suffix` (e.g., `train.lij`): Your target language training file.
  * `floresplus.dev.[src_suffix]-[tgt_suffix].[src_suffix].txt`: FLORES development source set.
  * `floresplus.dev.[src_suffix]-[tgt_suffix].[tgt_suffix].txt`: FLORES development target set.
  * `floresplus.devtest.[src_suffix]-[tgt_suffix].[src_suffix].txt`: FLORES evaluation source set.
  * `floresplus.devtest.[src_suffix]-[tgt_suffix].[tgt_suffix].txt`: FLORES evaluation target set.

### 3. Run and Monitor
Once these variables are set, the paths (`DRIVE_DIR`, `WORK_DIR`), data copying processes, tokenizer parameters, and model BOS token IDs automatically adapt dynamically to your language configuration without manual hardcoding.

In [ ]:
%pip install -q -U "transformers>=4.44,<5" datasets evaluate accelerate sentencepiece sacremoses sacrebleu ctranslate2 huggingface_hub pysbd tqdm

In [ ]:
from pathlib import Path
from datetime import datetime
import gc, json, logging, math, random, re, shutil, unicodedata, warnings

import evaluate
import numpy as np
import torch
from datasets import Dataset, DatasetDict
from google.colab import drive
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    DataCollatorForSeq2Seq,
    EarlyStoppingCallback,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)
from transformers.optimization import Adafactor, get_constant_schedule_with_warmup, get_linear_schedule_with_warmup

warnings.filterwarnings("ignore", message=r"Moving the following attributes in the config to the generation config.*")
logging.getLogger("transformers.modeling_utils").setLevel(logging.ERROR)

# Configure generic Source and Target languages
SRC_LANG, TGT_LANG = "eng_Latn", "lij_Latn"
src_code = SRC_LANG.split("_")[0]
tgt_code = TGT_LANG.split("_")[0]
lang_pair_suffix = f"{src_code}_{tgt_code}"

drive.mount("/content/drive")
DRIVE_DIR = Path(f"/content/drive/MyDrive/nllb_{lang_pair_suffix}")
WORK_DIR = Path(f"/content/nllb_{lang_pair_suffix}")
DATA_DIR = WORK_DIR / "data"
RUN_DIR = WORK_DIR / (f"run_1.3B_" + datetime.now().strftime("%Y%m%d_%H%M%S"))
FINAL_DIR = RUN_DIR / "final"
CT2_DIR = RUN_DIR / "ct2_float16"
ARCHIVE_PATH = WORK_DIR / f"{RUN_DIR.name}.tar.gz"

# Parametrized training, development, and test file lists
DATA_FILES = [
    f"train.{src_code}",
    f"train.{tgt_code}",
    f"floresplus.dev.{src_code}-{tgt_code}.{src_code}.txt",
    f"floresplus.dev.{src_code}-{tgt_code}.{tgt_code}.txt",
    f"floresplus.devtest.{src_code}-{tgt_code}.{src_code}.txt",
    f"floresplus.devtest.{src_code}-{tgt_code}.{tgt_code}.txt",
]

WORK_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)
RUN_DIR.mkdir(parents=True, exist_ok=True)

for name in DATA_FILES:
    src = DRIVE_DIR / name
    dst = DATA_DIR / name
    if not dst.exists() or dst.stat().st_size != src.stat().st_size:
        shutil.copy2(src, dst)

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
print(torch.cuda.get_device_name(0))
print("local run:", RUN_DIR)
print("drive archive target:", DRIVE_DIR / ARCHIVE_PATH.name)

In [ ]:
from huggingface_hub import login
from google.colab import userdata

login(token=userdata.get("HF_TOKEN"))

In [ ]:
DELETE_CHARS = {
    "\u00ad",  # soft hyphen
    "\u200b",  # zero-width space
    "\u200c",  # zero-width non-joiner
    "\u200d",  # zero-width joiner
    "\ufeff",  # zero-width no-break space / BOM
    "\u2060",  # word joiner
}

SPACE_CHARS = {
    "\u00a0",  # no-break space
    "\u1680",  # ogham space mark
    "\u2000",  # en quad
    "\u2001",  # em quad
    "\u2002",  # en space
    "\u2003",  # em space
    "\u2004",  # three-per-em space
    "\u2005",  # four-per-em space
    "\u2006",  # six-per-em space
    "\u2007",  # figure space
    "\u2008",  # punctuation space
    "\u2009",  # thin space
    "\u200a",  # hair space
    "\u202f",  # narrow no-break space
    "\u205f",  # medium mathematical space
    "\u3000",  # ideographic space
}

def clean(s):
    s = unicodedata.normalize("NFKC", s)
    s = "".join(c for c in s if unicodedata.category(c) not in {"Cc", "Cf", "Co"})
    for ch in DELETE_CHARS:
        s = s.replace(ch, "")
    for ch in SPACE_CHARS:
        s = s.replace(ch, " ")
    s = re.sub(r"[—–−]", "-", s)
    s = re.sub(r"['ʼ‘’‛`´ʹˈ]", "'", s)
    return re.sub(r"\s+", " ", s).strip()

def read_pair(src_path, tgt_path):
    src_lines = src_path.read_text(encoding="utf-8").splitlines()
    tgt_lines = tgt_path.read_text(encoding="utf-8").splitlines()
    if len(src_lines) != len(tgt_lines):
        raise ValueError(f"line mismatch: {src_path}={len(src_lines)} {tgt_path}={len(tgt_lines)}")
    out = []
    for a, b in zip(src_lines, tgt_lines):
        a, b = clean(a), clean(b)
        if len(a) >= 2 and len(b) >= 2:
            out.append((a, b))
    return out

# Dynamically locate training, development, and test file suffixes based on data configuration
# assuming filenames are configured in DATA_FILES list:
# [0]: train_src, [1]: train_tgt, [2]: dev_src, [3]: dev_tgt, [4]: test_src, [5]: test_tgt
train_pairs = read_pair(DATA_DIR / DATA_FILES[0], DATA_DIR / DATA_FILES[1])
dev_pairs = read_pair(DATA_DIR / DATA_FILES[2], DATA_DIR / DATA_FILES[3])
test_pairs = read_pair(DATA_DIR / DATA_FILES[4], DATA_DIR / DATA_FILES[5])

# De-duplicate train set from dev and test sets
train_pairs = [p for p in train_pairs if p not in set(dev_pairs) and p not in set(test_pairs)]
random.shuffle(train_pairs)

def rows(pairs):
    return [{"src": a, "tgt": b} for a, b in pairs]

ds = DatasetDict({
    "train": Dataset.from_list(rows(train_pairs)),
    "validation": Dataset.from_list(rows(dev_pairs)),
    "test": Dataset.from_list(rows(test_pairs)),
})
print(ds)

In [ ]:
MODEL_ID = "facebook/nllb-200-distilled-1.3B"
MAX_LENGTH = 164

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, src_lang=SRC_LANG, tgt_lang=TGT_LANG)
LANG_TOKEN_IDS = {x: tokenizer.convert_tokens_to_ids(x) for x in [SRC_LANG, TGT_LANG]}
print(LANG_TOKEN_IDS)
assert LANG_TOKEN_IDS[TGT_LANG] != tokenizer.unk_token_id

BEST_PARAMS = {"clip_threshold": 1.0, "lr": 0.00015, "warmup_ratio": 0.03, "weight_decay": 0.001}

def banner(text):
    print("\n" + "=" * 88)
    print(text)
    print("=" * 88)

def fmt_params(params):
    return json.dumps(params, sort_keys=True)

def tokenize(batch):
    tokenizer.src_lang = SRC_LANG
    tokenizer.tgt_lang = TGT_LANG
    x = tokenizer(batch["src"], text_target=batch["tgt"], max_length=MAX_LENGTH, truncation=True)
    x["labels"] = [[t if t != tokenizer.pad_token_id else -100 for t in y] for y in x["labels"]]
    return x

tokenized_ds = ds.map(tokenize, batched=True, batch_size=512, remove_columns=ds["train"].column_names)
first = [x for x in tokenized_ds["train"][0]["labels"] if x != -100][:8]
print(tokenizer.convert_ids_to_tokens(first))
assert first[0] == LANG_TOKEN_IDS[TGT_LANG]

bleu = evaluate.load("sacrebleu")

def metrics(pred):
    pred_ids, labels = pred
    if isinstance(pred_ids, tuple):
        pred_ids = pred_ids[0]
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    hyp = [x.strip() for x in tokenizer.batch_decode(pred_ids, skip_special_tokens=True)]
    ref = [[x.strip()] for x in tokenizer.batch_decode(labels, skip_special_tokens=True)]
    return {"bleu": bleu.compute(predictions=hyp, references=ref)["score"]}

def fresh_model():
    m = AutoModelForSeq2SeqLM.from_pretrained(MODEL_ID)
    m.config.use_cache = False
    m.config.max_length = None
    m.generation_config.forced_bos_token_id = LANG_TOKEN_IDS[TGT_LANG]
    m.generation_config.max_length = MAX_LENGTH
    return m.to("cuda")

def optimizers_for(m, params, total_steps):
    opt = Adafactor(
        [p for p in m.parameters() if p.requires_grad],
        scale_parameter=False,
        relative_step=False,
        lr=params["lr"],
        clip_threshold=params["clip_threshold"],
        weight_decay=params["weight_decay"],
    )
    warmup = max(1, int(total_steps * params["warmup_ratio"]))
    sched = get_linear_schedule_with_warmup(opt, num_warmup_steps=warmup, num_training_steps=total_steps)
    return opt, sched

def trainer_for(m, params, out):
    steps = math.ceil(len(tokenized_ds["train"]) / 64)
    total_steps = steps * 2
    eval_steps = max(1, steps // 4)

    args = Seq2SeqTrainingArguments(
        output_dir=str(out),
        eval_strategy="steps",
        eval_steps=eval_steps,
        save_strategy="steps",
        save_steps=eval_steps,
        save_total_limit=1,
        logging_steps=100,
        per_device_train_batch_size=64,
        per_device_eval_batch_size=64,
        num_train_epochs=3,
        bf16=True,
        tf32=True,
        gradient_checkpointing=False,
        group_by_length=True,
        predict_with_generate=True,
        generation_max_length=MAX_LENGTH,
        generation_num_beams=4,
        load_best_model_at_end=True,
        metric_for_best_model="bleu",
        greater_is_better=True,
        report_to="none",
        save_safetensors=True,
    )
    return Seq2SeqTrainer(
        model=m,
        args=args,
        train_dataset=tokenized_ds["train"],
        eval_dataset=tokenized_ds["validation"],
        data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer, model=m, label_pad_token_id=-100, pad_to_multiple_of=8),
        compute_metrics=metrics,
        processing_class=tokenizer,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
        optimizers=optimizers_for(m, params, total_steps),
    )

def print_dev_examples(m, title):
    banner(title)
    m.eval()
    for idx in [17, 231, 456]:
        row = ds["validation"][idx]
        x = tokenizer(row["src"], return_tensors="pt", truncation=True, max_length=MAX_LENGTH).to("cuda")
        with torch.no_grad():
            y = m.generate(**x, forced_bos_token_id=LANG_TOKEN_IDS[TGT_LANG], max_length=MAX_LENGTH, num_beams=4)
        print("SRC:", row["src"])
        print("REF:", row["tgt"])
        print("HYP:", tokenizer.decode(y[0], skip_special_tokens=True))
        print()

banner("BASE MODEL VALIDATION")
base_model = fresh_model()
base_trainer = Seq2SeqTrainer(
    model=base_model,
    args=Seq2SeqTrainingArguments(
        output_dir=str(RUN_DIR / "baseline"),
        per_device_eval_batch_size=64,
        predict_with_generate=True,
        generation_max_length=MAX_LENGTH,
        generation_num_beams=4,
        report_to="none",
    ),
    eval_dataset=tokenized_ds["validation"],
    data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer, model=base_model, label_pad_token_id=-100, pad_to_multiple_of=8),
    compute_metrics=metrics,
    processing_class=tokenizer,
)
print("initial validation:", base_trainer.evaluate())
print_dev_examples(base_model, "initial dev examples")
del base_trainer, base_model
gc.collect(); torch.cuda.empty_cache()

banner("TRAINING FIXED BEST PARAMS")
print("params:", fmt_params(BEST_PARAMS))
model = fresh_model()
trainer = trainer_for(model, BEST_PARAMS, RUN_DIR / "checkpoints")
trainer.train(resume_from_checkpoint=False)
final_metrics = trainer.evaluate()
print("final validation:", final_metrics)
trainer.save_model(str(FINAL_DIR))
tokenizer.save_pretrained(str(FINAL_DIR))
chosen = {"source": "fixed_best_params", "score": final_metrics["eval_bleu"], "params": BEST_PARAMS, "model_dir": str(FINAL_DIR)}
(RUN_DIR / "chosen_final.json").write_text(json.dumps(chosen, indent=2), encoding="utf-8")
assert FINAL_DIR.exists(), FINAL_DIR
print_dev_examples(model, "final dev examples")
print("saved final model:", FINAL_DIR)
print("chosen final:", json.dumps(chosen, indent=2))

In [ ]:
DEVICE = "cuda"

eval_model_dir = FINAL_DIR
print("evaluating final model:", eval_model_dir)
model = AutoModelForSeq2SeqLM.from_pretrained(str(eval_model_dir)).to(DEVICE)
model.config.use_cache = False
model.config.max_length = None
model.generation_config.forced_bos_token_id = LANG_TOKEN_IDS[TGT_LANG]
model.generation_config.max_length = MAX_LENGTH
model.eval()

def translate(texts, bs=64):
    tokenizer.src_lang = SRC_LANG
    out = []
    for i in range(0, len(texts), bs):
        x = tokenizer(texts[i:i+bs], return_tensors="pt", padding=True, truncation=True, max_length=MAX_LENGTH).to(DEVICE)
        with torch.no_grad():
            y = model.generate(**x, forced_bos_token_id=LANG_TOKEN_IDS[TGT_LANG], max_length=MAX_LENGTH, num_beams=4)
        out += tokenizer.batch_decode(y, skip_special_tokens=True)
    return out

pred = translate(ds["test"]["src"])
score = bleu.compute(predictions=pred, references=[[x] for x in ds["test"]["tgt"]])["score"]
(RUN_DIR / "test_sacrebleu.json").write_text(json.dumps({"bleu": score, "n": len(pred), "model_dir": str(eval_model_dir)}, indent=2), encoding="utf-8")
print(f"test BLEU: {score:.2f}")
print("Good morning, how are you? ->", translate(["Good morning, how are you?"], bs=1)[0])

y = model.generate(**tokenizer(["Good morning, how are you?"], return_tensors="pt").to(DEVICE), forced_bos_token_id=LANG_TOKEN_IDS[TGT_LANG], max_length=64)
print(tokenizer.convert_ids_to_tokens(y[0].detach().cpu().tolist()[:12]))

In [ ]:
from huggingface_hub import create_repo

HUB_MODEL_ID = "ConseggioLigure/nllb1.3B-eng-lld"

model.save_pretrained(str(FINAL_DIR))
tokenizer.save_pretrained(str(FINAL_DIR))
create_repo(HUB_MODEL_ID, private = True, exist_ok = True)
model.push_to_hub(HUB_MODEL_ID)
tokenizer.push_to_hub(HUB_MODEL_ID)

In [ ]:
convert_model_dir = FINAL_DIR
print("converting final model:", convert_model_dir)
if CT2_DIR.exists():
    shutil.rmtree(CT2_DIR)
!ct2-transformers-converter --model "{convert_model_dir}" --output_dir "{CT2_DIR}" --quantization float16
tokenizer.save_pretrained(str(CT2_DIR))
print(CT2_DIR)

if ARCHIVE_PATH.exists():
    ARCHIVE_PATH.unlink()
!tar -czf "{ARCHIVE_PATH}" -C "{WORK_DIR}" "{RUN_DIR.name}"
shutil.copy2(ARCHIVE_PATH, DRIVE_DIR / ARCHIVE_PATH.name)
print("copied archive to Drive:", DRIVE_DIR / ARCHIVE_PATH.name)

## Inference

We reload the model from HF in case this part of the notebook is being run in a new session, when the `model` variable is no longer in session.

In [ ]:
import os, subprocess, sys
from pathlib import Path
import pysbd
from tqdm.auto import tqdm
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from huggingface_hub import login
try:
    from google.colab import userdata
except Exception:
    userdata = None

WMT26_INPUT = Path('/content/drive/MyDrive/Datasets/WMT26/wmt26_genmt_blindset.jsonl')
WMT26_OUT_DIR = Path('/content/drive/MyDrive/Datasets/WMT26/outputs')
WMT26_OUTPUT = WMT26_OUT_DIR / f'wmt26_genmt_blindset.lld_Latn.nllb.jsonl'
WMT26_OUT_DIR.mkdir(parents=True, exist_ok=True)

assert WMT26_INPUT.exists(), WMT26_INPUT
print(WMT26_OUTPUT)

In [ ]:
SRC_LANG = "eng_Latn"
TGT_LANG = "lmo_Latn"
MODEL_ID = "ConseggioLigure/nllb1.3B-eng-lld"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

if userdata is not None:
    token = userdata.get("HF_TOKEN")
    if token:
        login(token=token)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, src_lang=SRC_LANG, tgt_lang=TGT_LANG)
model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
    low_cpu_mem_usage=True,
).to(DEVICE)
model.eval()
model.config.use_cache = True
LANG_TOKEN_IDS = {lang: tokenizer.convert_tokens_to_ids(lang) for lang in [SRC_LANG, TGT_LANG]}
assert LANG_TOKEN_IDS[TGT_LANG] != tokenizer.unk_token_id
print(DEVICE, LANG_TOKEN_IDS)

In [ ]:
FENCED_JSON_RE = re.compile(r"\A(?P<prefix>```json[^\n\r]*(?:\r\n|\n|\r))(?P<body>.*)(?P<suffix>(?:\r\n|\n|\r)```[ \t]*(?:\r\n|\n|\r)?)\Z", re.S)
NEWLINE_RE = re.compile(r"(\r\n|\n|\r)")
HTML_TAG_RE = re.compile(r"(<[A-Za-z][^<>]*>|</[A-Za-z][^<>]*>)")
PLACEHOLDER_RE = re.compile(r"(\{\{[^{}]+\}\})")
MAX_WMT_INPUT_LENGTH = 512
MAX_WMT_NEW_TOKENS = 256
WMT_BATCH_SIZE = 32
WMT_NUM_BEAMS = 4

class TranslationPlanner:
    def __init__(self, segmenter):
        self.segmenter = segmenter
        self.items = []

    def add(self, text):
        if not any(char.isalpha() for char in text):
            return text
        self.items.append(text)
        return len(self.items) - 1

    def plan_text(self, text):
        plan = []
        for part in NEWLINE_RE.split(text):
            if not part:
                continue
            plan.extend([part] if NEWLINE_RE.fullmatch(part) else self.plan_line(part))
        return plan

    def plan_line(self, line):
        if not line.strip() or not any(char.isalpha() for char in line):
            return [line]
        if PLACEHOLDER_RE.search(line):
            plan = []
            for part in PLACEHOLDER_RE.split(line):
                if part:
                    plan.extend([part] if PLACEHOLDER_RE.fullmatch(part) else self.plan_line(part))
            return plan

        plan = []
        last_end = 0
        for span in self.segmenter.segment(line):
            plan.append(line[last_end:span.start])
            sentence = line[span.start:span.end]
            leading_len = len(sentence) - len(sentence.lstrip())
            trailing_len = len(sentence) - len(sentence.rstrip())
            leading = sentence[:leading_len]
            trailing = sentence[len(sentence) - trailing_len:] if trailing_len else ""
            core_end = len(sentence) - trailing_len if trailing_len else len(sentence)
            core = sentence[leading_len:core_end]
            plan.extend([leading, self.add(core), trailing] if core and any(char.isalpha() for char in core) else [sentence])
            last_end = span.end
        plan.append(line[last_end:])
        return plan

    def plan_html_text(self, text):
        plan = []
        for part in HTML_TAG_RE.split(text):
            if part:
                plan.extend([part] if HTML_TAG_RE.fullmatch(part) else self.plan_text(part))
        return plan


def render_plan(plan, translations):
    return "".join(translations[item] if isinstance(item, int) else item for item in plan)


@torch.inference_mode()
def translate_items(texts):
    tokenizer.src_lang = SRC_LANG
    out = []
    for i in range(0, len(texts), WMT_BATCH_SIZE):
        x = tokenizer(
            texts[i:i + WMT_BATCH_SIZE],
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_WMT_INPUT_LENGTH,
        ).to(DEVICE)
        y = model.generate(
            **x,
            forced_bos_token_id=LANG_TOKEN_IDS[TGT_LANG],
            num_beams=WMT_NUM_BEAMS,
            max_new_tokens=MAX_WMT_NEW_TOKENS,
        )
        out.extend(t.strip() for t in tokenizer.batch_decode(y, skip_special_tokens=True))
    return out


def read_json_string(text, start):
    index = start + 1
    escaped = False
    while index < len(text):
        char = text[index]
        if escaped:
            escaped = False
        elif char == "\\":
            escaped = True
        elif char == '"':
            raw = text[start:index + 1]
            return raw, json.loads(raw), index + 1
        index += 1
    raise ValueError("Unterminated JSON string")


def json_context_expects_value(stack):
    if not stack:
        return True
    context, state = stack[-1]
    return (context == "object" and state == "value") or (context == "array" and state == "value_or_end")


def json_mark_value_read(stack):
    if stack:
        context, _ = stack[-1]
        stack[-1] = (context, "comma_or_end")


def translate_json_values(text, planner):
    stack, output, plans = [], [], []
    index = 0
    while index < len(text):
        char = text[index]
        if char == '"':
            raw, value, next_index = read_json_string(text, index)
            if stack and stack[-1] == ("object", "key_or_end"):
                output.append(raw)
                stack[-1] = ("object", "colon")
            elif json_context_expects_value(stack):
                output.append(("string", len(plans)))
                plans.append(planner.plan_text(value))
                json_mark_value_read(stack)
            else:
                output.append(raw)
            index = next_index
            continue

        output.append(char)
        if char == "{":
            if json_context_expects_value(stack):
                json_mark_value_read(stack)
            stack.append(("object", "key_or_end"))
        elif char == "[":
            if json_context_expects_value(stack):
                json_mark_value_read(stack)
            stack.append(("array", "value_or_end"))
        elif char == "}" and stack and stack[-1][0] == "object":
            stack.pop()
        elif char == "]" and stack and stack[-1][0] == "array":
            stack.pop()
        elif char == ":" and stack and stack[-1] == ("object", "colon"):
            stack[-1] = ("object", "value")
        elif char == "," and stack:
            context, state = stack[-1]
            if context == "object" and state == "comma_or_end":
                stack[-1] = ("object", "key_or_end")
            elif context == "array" and state == "comma_or_end":
                stack[-1] = ("array", "value_or_end")
        index += 1

    translations = translate_items(planner.items)
    rendered = [json.dumps(render_plan(plan, translations), ensure_ascii=False) for plan in plans]
    return "".join(rendered[item[1]] if isinstance(item, tuple) else item for item in output)

In [ ]:
wmt_rows = []
with WMT26_INPUT.open(encoding='utf-8') as f:
    for line in f:
        row = json.loads(line)
        if row.get('tgt_lang') == "lld_Latn":
            wmt_rows.append(row)

assert len(wmt_rows) == 198
segmenter = pysbd.Segmenter(language='en', clean=False, char_span=True)
mode_counts = {}

with WMT26_OUTPUT.open('w', encoding='utf-8') as out:
    for row in tqdm(wmt_rows, desc=f'WMT26 {TGT_LANG}'):
        source_doc = row['source_doc']
        instruction = row['instruction']
        if 'software data' in instruction:
            mode = 'json'
        elif 'Maintain the HTML formatting' in instruction:
            mode = 'html'
        else:
            mode = 'text'
        mode_counts[mode] = mode_counts.get(mode, 0) + 1

        planner = TranslationPlanner(segmenter)
        if mode == 'json':
            match = FENCED_JSON_RE.match(source_doc)
            if not match:
                raise ValueError(row['doc_id'])
            body = translate_json_values(match.group('body'), planner)
            hypothesis = f"{match.group('prefix')}{body}{match.group('suffix')}"
        elif mode == 'html':
            plan = planner.plan_html_text(source_doc)
            hypothesis = render_plan(plan, translate_items(planner.items))
        else:
            plan = planner.plan_text(source_doc)
            hypothesis = render_plan(plan, translate_items(planner.items))

        output_row = dict(row)
        output_row['hypothesis'] = hypothesis
        out.write(json.dumps(output_row, ensure_ascii=False) + '\n')

print(mode_counts)
print(WMT26_OUTPUT)

In [ ]:
source = {row["doc_id"]: row for row in wmt_rows}
seen, errors = set(), []

def parse_or_count_jsonish(body):
    try:
        obj = json.loads(body)
        if isinstance(obj, dict):
            return "dict", set(obj), len(obj)
        if isinstance(obj, list):
            return "list", None, len(obj)
        return type(obj).__name__, None, 1
    except json.JSONDecodeError:
        text = body.strip()
        if text.startswith("["):
            return "list", None, len(re.findall(r'^\s*"', body, flags=re.M))
        if text.startswith("{"):
            return "dict", set(re.findall(r'^\s*"((?:\\.|[^"\\])+)"\s*:', body, flags=re.M)), None
        raise

with WMT26_OUTPUT.open(encoding="utf-8") as f:
    output_rows = [json.loads(line) for line in f]

for row in output_rows:
    required_keys = set(source[row["doc_id"]]) | {"hypothesis"}
    if set(row) != required_keys:
        errors.append((row.get("doc_id"), "bad keys"))
    if row["doc_id"] in seen:
        errors.append((row["doc_id"], "duplicate"))
    seen.add(row["doc_id"])

    src = source[row["doc_id"]]
    for key, value in src.items():
        if row.get(key) != value:
            errors.append((row["doc_id"], f"source field changed: {key}"))

    hyp = row["hypothesis"]
    if row["tgt_lang"] != "lld_Latn":
        errors.append((row["doc_id"], "bad tgt_lang"))

    if "software data" in src["instruction"]:
        src_match = FENCED_JSON_RE.match(src["source_doc"])
        hyp_match = FENCED_JSON_RE.match(hyp)
        if not src_match or not hyp_match:
            errors.append((row["doc_id"], "json fence"))
        else:
            src_kind, src_keys, src_count = parse_or_count_jsonish(src_match.group("body"))
            hyp_kind, hyp_keys, hyp_count = parse_or_count_jsonish(hyp_match.group("body"))
            if src_kind != hyp_kind:
                errors.append((row["doc_id"], "json root"))
            if src_count is not None and hyp_count is not None and src_count != hyp_count:
                errors.append((row["doc_id"], "json count"))
            if src_keys is not None and hyp_keys is not None and src_keys != hyp_keys:
                errors.append((row["doc_id"], "json keys"))
        if sorted(set(re.findall(r"\{\{[^{}]+\}\}", src["source_doc"]))) != sorted(set(re.findall(r"\{\{[^{}]+\}\}", hyp))):
            errors.append((row["doc_id"], "placeholders"))

    if "Maintain the HTML formatting" in src["instruction"]:
        if src["source_doc"].count("<p>") != hyp.count("<p>"):
            errors.append((row["doc_id"], "p count"))
        if re.findall(r"</?[A-Za-z][^<>]*>", src["source_doc"]) != re.findall(r"</?[A-Za-z][^<>]*>", hyp):
            errors.append((row["doc_id"], "tag order"))

assert len(output_rows) == len(wmt_rows) == 198
assert seen == set(source)
assert not errors, errors[:20]
print("local validation ok")

In [ ]:
checker = WMT26_OUT_DIR / 'genmt_check_alignment.py'
stub_dir = WMT26_OUT_DIR / 'checker_stubs'
stub_dir.mkdir(exist_ok=True)
(stub_dir / 'ipdb.py').write_text('', encoding='utf-8')
if not checker.exists():
    !curl -L https://raw.githubusercontent.com/wmt-conference/wmt-collect-translations/main/genmt_check_alignment.py -o "{checker}"

result = subprocess.run(
    [sys.executable, str(checker), '--source_path', str(WMT26_INPUT), '--translation_path', str(WMT26_OUTPUT)],
    env={**os.environ, 'PYTHONPATH': str(stub_dir)},
    text=True,
    capture_output=True,
)
print(result.stdout)
print(result.stderr)
assert result.returncode == 0